In [ ]:
#Install needed libraries and dependencies
!pip install "torch==2.5.1" "torchaudio==2.5.1" "torchvision==0.20.1" "torchcodec==0.1.0"
!pip install "vllm==0.6.4.post1" "transformers==4.46.3" "tokenizers==0.20.3" accelerate pandas datasets scipy sentence-transformers faiss-cpu

In [ ]:
#RAG Retrieval
import os
import gc
import pickle
import pandas as pd
import torch
import faiss
from sentence_transformers import SentenceTransformer
from google.colab import drive

drive.mount('/content/drive',force_remount=True)

def flush_memory():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

VOTING_CSV_PATH = "PATH_WHERE_TIER_3_OUTPUT_SAVED"
TEXT_CSV_PATH = "PATH_WHERE_TIER_2_OUTPUT_SAVED"
INDEX_PATH = "PATH_WHERE_RAG_INDEX_SAVED"
METADATA_PATH = "PATH_WHERE_RAG_METADATA_SAVED"

CACHE_PATH = "PATH_WHERE_CACHE_FILES_WILL_BE_STORED" #Ensures this cell only needs to run once
os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)

print("Loading datasets...")
df_voting = pd.read_csv(VOTING_CSV_PATH)
df_text = pd.read_csv(TEXT_CSV_PATH)

if len(df_voting) != len(df_text):
    raise ValueError(f"Length mismatch: Voting CSV has {len(df_voting)} rows, Text CSV has {len(df_text)} rows.")

text_columns = ['question_en', 'rationale_Logical', 'option1', 'option2', 'option3', 'option4']
for col in text_columns:
    if col not in df_text.columns:
        raise ValueError(f"Missing required column '{col}' in TEXT_CSV_PATH.")
    df_voting[col] = df_text[col]

print("Loading FAISS Index and Metadata...")
index = faiss.read_index(INDEX_PATH)
with open(METADATA_PATH, 'rb') as f:
    corpus_data = pickle.load(f)

print("Loading Retriever (BGE-M3)...")
retriever = SentenceTransformer("BAAI/bge-m3").to("cuda")

prompts = []
print("Building Prompts and fetching context...")

for idx, row in df_voting.iterrows():
    q_emb = retriever.encode([row['question_en']], normalize_embeddings=True)
    similarities, doc_indices = index.search(q_emb, k=1)
    top_score = similarities[0][0]
    top_doc = corpus_data[doc_indices[0][0]]

    prompt_start = f"You are a strict and knowledgable evaluator. Analyze the given information and provide the final correct option letter (A, B, C, or D).\nEnglish Question: {row['question_en']}\n"

    if top_score >= 0.85:
        prompt_context = f"Knowledge Context: {top_doc}\n"
    else:
        prompt_context = ""

    prompt_end = f"Proposed Logical Rationale: {row['rationale_Logical']}\nOptions:\nA) {row['option1']}\nB) {row['option2']}\nC) {row['option3']}\nD) {row['option4']}\nCurrent Ensemble Prediction: {row['Final_Voted_Pred']}\n\nRefined Final Answer (Letter only):"

    critic_prompt = prompt_start + prompt_context + prompt_end
    prompts.append(critic_prompt)

df_voting['critic_prompt'] = prompts

df_voting.to_pickle(CACHE_PATH)
print(f"Cached prompts successfully saved to {CACHE_PATH}")

print("Purging BGE-M3 from memory...")
del retriever, index, corpus_data
flush_memory()

In [ ]:
#Critic
import os
import gc
import math
import pandas as pd
import torch
from vllm import LLM, SamplingParams

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

def flush_memory():
    gc.collect()
    torch.cuda.empty_cache()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

# Paths
CACHE_PATH = "PATH_WHERE_CACHE_IS_STORED"
OUTPUT_PATH = "PATH_WHERE_OUTPUT_WILL_BE_STORED"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

print("Loading cached prompts...")
if not os.path.exists(CACHE_PATH):
    raise FileNotFoundError("Cache file not found. Run Cell 1 first.")

df_voting = pd.read_pickle(CACHE_PATH)
prompts = df_voting['critic_prompt'].tolist()

print("Loading Critic via vLLM...")
critic_id = "sarvamai/sarvam-1" #Critic Model

llm = LLM(
    model=critic_id,
    dtype="half",
    gpu_memory_utilization=0.85,
    max_model_len=4096,
    enforce_eager=True,
    disable_custom_all_reduce=True,
    trust_remote_code=True
)

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=1,
    logprobs=5
)

print("Batch generating Critic answers...")
outputs = llm.generate(prompts, sampling_params, use_tqdm=True)

print("Applying probability gates and formatting output...")
rag_preds = []
critic_overrides = 0
confidence_fallbacks = 0

for i, output in enumerate(outputs):
    ensemble_pred = df_voting.iloc[i]['Final_Voted_Pred']

    raw_text = output.outputs[0].text.strip().upper()
    valid_options = ["A", "B", "C", "D"]
    parsed_critic_result = next((char for char in raw_text if char in valid_options), None)

    try:
        first_token_logprobs = output.outputs[0].logprobs[0]
        gen_token_id = output.outputs[0].token_ids[0]
        logprob_val = first_token_logprobs[gen_token_id].logprob
        confidence_score = math.exp(logprob_val)
    except (IndexError, KeyError, TypeError):
        confidence_score = 0.0

    if parsed_critic_result and confidence_score >= 0.60:
        rag_preds.append(parsed_critic_result)
        critic_overrides += 1
    else:
        rag_preds.append(ensemble_pred)
        confidence_fallbacks += 1

In [ ]:
#Final Result
df_voting['RAG_Pred'] = rag_preds
df_voting['Final_Correct'] = df_voting['RAG_Pred'].astype(str) == df_voting['True_Label'].astype(str)

columns_to_drop = ['question_en', 'rationale_Logical', 'option1', 'option2', 'option3', 'option4', 'critic_prompt']
df_final = df_voting.drop(columns=columns_to_drop)

print(f"\nCritic confidently answered (>=0.60): {critic_overrides} times.")
print(f"Critic rejected due to low confidence (<0.60): {confidence_fallbacks} times.")

df_final.to_csv(OUTPUT_PATH, index=False)
print(f"Final results successfully saved to {OUTPUT_PATH}")

del llm
flush_memory()

In [ ]:
#Accuracy Report
import pandas as pd
import os

RESULTS_CSV_PATH = "PATH_WHERE_TIER_4_RESULT_STORED"

if not os.path.exists(RESULTS_CSV_PATH):
    raise FileNotFoundError(f"File not found: {RESULTS_CSV_PATH}. Please ensure Cell 2 completed successfully.")

df = pd.read_csv(RESULTS_CSV_PATH)
total_questions = len(df)

rag_correct = df['Final_Correct'].astype(str).str.lower().eq('true').sum()
voting_correct = df['Is_Correct'].astype(str).str.lower().eq('true').sum()

rag_accuracy = (rag_correct / total_questions) * 100
voting_accuracy = (voting_correct / total_questions) * 100
accuracy_change = rag_accuracy - voting_accuracy

print("=" * 40)
print("ACCURACY REPORT")
print("=" * 40)
print(f"Total Questions Evaluated: {total_questions}\n")

print(f"Original Voting Accuracy (Stage 3): {voting_accuracy:.2f}% ({voting_correct}/{total_questions})")
print(f"Final RAG Critic Accuracy (Stage 4): {rag_accuracy:.2f}% ({rag_correct}/{total_questions})\n")

if accuracy_change > 0:
    print(f"Net Change: +{accuracy_change:.2f}% (Improvement)")
elif accuracy_change < 0:
    print(f"Net Change: {accuracy_change:.2f}% (Regression)")
else:
    print("Net Change: 0.00% (No change in overall accuracy)")
print("=" * 40)

ACCURACY REPORT
Total Questions Evaluated: 4099

Original Voting Accuracy (Stage 3): 51.26% (2101/4099)
Final RAG Critic Accuracy (Stage 4): 51.40% (2107/4099)

Net Change: +0.15% (Improvement)
